# Lab 1 vs Lab 2 — Comparación de modelos

<a href="https://colab.research.google.com/github/lgonperez3-glitch/MLSM-labs/blob/main/Lab2/lab1vslab2_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este notebook entrena la ANN ganadora de Lab 1 y prepara una comparación de sus resultados con Lab 2 Task 1 (CNN) y Task 2 (VGG16).

**Partición de Lab 1:** mismo procedimiento que los notebooks de Lab 2: 80/20 estratificado y, dentro del 80 % de desarrollo, 80/20 estratificado. Resultado aproximado: 64 % train, 16 % validation y 20 % test. Semilla 42.

**Datos:** CSV de características tabulares de GTZAN. Las redes de Lab 2 utilizan imágenes de espectrogramas. La misma semilla y proporciones no garantizan las mismas canciones en test, porque las listas de muestras y sus órdenes son diferentes. Por tanto, esta comparación es de rendimiento bajo un protocolo semejante, no una comparación emparejada sobre idénticas muestras.

El notebook no inventa ni reutiliza accuracies antiguas de Lab 2. Al final permite introducir las accuracies de test de las ejecuciones que quieras comparar.

## 1. Montar Google Drive y configurar rutas

El CSV se lee desde la ruta que has indicado para Lab 1. No es necesario cargar las imágenes para entrenar la ANN.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

CSV_PATH = Path("/content/drive/MyDrive/MSLS/Data/features_30_sec.csv")
OUTPUT_DIR = Path("/content/drive/MyDrive/MSLS/Results/lab1vslab2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"No se encuentra el CSV: {CSV_PATH}\n"
        "Comprueba que el archivo está en MyDrive/MSLS/Data/."
    )

print("CSV:", CSV_PATH)
print("Resultados:", OUTPUT_DIR)

## 2. Importaciones y reproducibilidad

Fijamos la semilla utilizada en Lab 2. Esto hace reproducible la partición siempre que el CSV, el orden de sus filas y las etiquetas no cambien.

In [ ]:
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_fscore_support
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 3. Cargar y preparar el CSV

La columna `label` contiene el género. El modelo recibe únicamente columnas numéricas. Las columnas de texto, como `filename`, no se utilizan como características.

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Dimensiones iniciales:", df.shape)

if "label" not in df.columns:
    raise ValueError("No existe la columna obligatoria 'label'.")

df = df.dropna(subset=["label"]).copy()
y_text = df["label"].astype(str)

X = df.drop(columns=["label"], errors="ignore")
X = X.select_dtypes(include=[np.number]).copy()

if X.empty:
    raise ValueError("No se encontraron columnas numéricas para entrenar.")

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = label_encoder.classes_

print("Canciones:", len(X))
print("Características numéricas:", X.shape[1])
print("Clases:", list(class_names))
display(y_text.value_counts().sort_index().rename("número de canciones").to_frame())

## 4. Partición 64/16/20 — igual procedimiento que Lab 2

Primero se reserva el 20 % para test. Después, el 20 % del 80 % restante se reserva para validación. Ambas divisiones son estratificadas y usan `random_state=42`.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.20, random_state=SEED, stratify=y_dev
)

split_summary = pd.DataFrame({
    "Conjunto": ["Train", "Validation", "Test", "Total"],
    "Muestras": [len(X_train), len(X_val), len(X_test), len(X)],
})
split_summary["Porcentaje"] = 100 * split_summary["Muestras"] / len(X)
display(split_summary)

# Distribución por clase para comprobar la estratificación.
distribution = pd.DataFrame({
    "Train": pd.Series(y_train).value_counts(normalize=True),
    "Validation": pd.Series(y_val).value_counts(normalize=True),
    "Test": pd.Series(y_test).value_counts(normalize=True),
}).fillna(0)
distribution.index = [class_names[i] for i in distribution.index]
display((100 * distribution).round(2))

## 5. Definir la ANN ganadora de Lab 1

Se conservan los hiperparámetros seleccionados en Lab 1:
- Capas ocultas: (128, 64)
- `alpha=0.001`
- `learning_rate_init=0.001`
- Optimizador: Adam
- `max_iter=10000`
- Semilla: 42

El preprocesamiento se ajusta dentro del pipeline. Así, la mediana, el escalado y la selección de características se calculan únicamente con el conjunto de entrenamiento y no filtran información de validación/test.

In [ ]:
ann_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("variance", VarianceThreshold(threshold=0.0)),
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k="all")),
    ("model", MLPClassifier(
        hidden_layer_sizes=(128, 64),
        solver="adam",
        learning_rate_init=0.001,
        alpha=0.001,
        max_iter=10000,
        random_state=SEED,
        early_stopping=False,
        n_iter_no_change=15,
        tol=1e-3
    ))
])

display(ann_pipeline)

## 6. Entrenar la ANN

El modelo se entrena solo con train. Validation se reserva para observar generalización; no se utiliza para ajustar pesos porque el MLP conserva `early_stopping=False`, como en la configuración ganadora original.

In [ ]:
ann_pipeline.fit(X_train, y_train)
mlp = ann_pipeline.named_steps["model"]

print("Entrenamiento completado.")
print("Iteraciones realizadas:", mlp.n_iter_)
print("Loss final:", mlp.loss_)
print("Convergencia indicada por sklearn:", mlp.n_iter_ < mlp.max_iter)

## 7. Curva de pérdida del entrenamiento

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(mlp.loss_curve_, label="Train loss")
plt.xlabel("Iteración")
plt.ylabel("Loss")
plt.title("Lab 1 — Pérdida durante el entrenamiento")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Métricas de train, validation y test

La métrica principal para la comparación con Lab 2 es la accuracy de test. Train y validation ayudan a diagnosticar sobreajuste.

In [ ]:
def evaluate_split(name, features, labels):
    pred = ann_pipeline.predict(features)
    return {
        "Split": name,
        "N": len(labels),
        "Accuracy": accuracy_score(labels, pred),
        "Precision macro": precision_recall_fscore_support(
            labels, pred, average="macro", zero_division=0
        )[0],
        "Recall macro": precision_recall_fscore_support(
            labels, pred, average="macro", zero_division=0
        )[1],
        "F1 macro": precision_recall_fscore_support(
            labels, pred, average="macro", zero_division=0
        )[2],
    }

ann_metrics_df = pd.DataFrame([
    evaluate_split("Train", X_train, y_train),
    evaluate_split("Validation", X_val, y_val),
    evaluate_split("Test", X_test, y_test),
])
display(ann_metrics_df.round(4))

ann_test_accuracy = float(ann_metrics_df.loc[
    ann_metrics_df["Split"] == "Test", "Accuracy"
].iloc[0])

## 9. Informe de clasificación y matrices de confusión

La diagonal de la matriz representa las predicciones correctas. La matriz normalizada permite comparar el recall de cada género independientemente de cuántas muestras tenga.

In [ ]:
y_test_pred = ann_pipeline.predict(X_test)

print(classification_report(
    y_test, y_test_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0,
    digits=4
))

cm = confusion_matrix(y_test, y_test_pred, labels=np.arange(len(class_names)))
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap="Blues", xticks_rotation=45, values_format="d", colorbar=False
)
ax.set_title("Lab 1 — Matriz de confusión (test)")
plt.tight_layout()
plt.show()

cm_norm = confusion_matrix(
    y_test, y_test_pred,
    labels=np.arange(len(class_names)),
    normalize="true"
)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
    xticklabels=class_names, yticklabels=class_names, ax=ax
)
ax.set_xlabel("Género predicho")
ax.set_ylabel("Género real")
ax.set_title("Lab 1 — Matriz de confusión normalizada")
plt.tight_layout()
plt.show()

## 10. Guardar resultados de Lab 1

Se guardan las métricas, predicciones, nombres de clase y curva de pérdida en Drive para poder consultar los resultados después.

In [ ]:
lab1_result = {
    "model": "MLPClassifier",
    "seed": SEED,
    "split": {"train": 0.64, "validation": 0.16, "test": 0.20},
    "n_samples": int(len(X)),
    "n_train": int(len(X_train)),
    "n_validation": int(len(X_val)),
    "n_test": int(len(X_test)),
    "class_names": [str(x) for x in class_names],
    "test_accuracy": ann_test_accuracy,
    "metrics": ann_metrics_df.to_dict(orient="records"),
    "hyperparameters": {
        "hidden_layer_sizes": [128, 64],
        "alpha": 0.001,
        "learning_rate_init": 0.001,
        "solver": "adam",
        "max_iter": 10000,
        "early_stopping": False,
        "n_iter_no_change": 15,
        "tol": 0.001
    }
}

with open(OUTPUT_DIR / "lab1_metrics.json", "w") as f:
    json.dump(lab1_result, f, indent=2)

np.savez(
    OUTPUT_DIR / "lab1_test_predictions.npz",
    y_true=y_test,
    y_pred=y_test_pred,
    class_names=class_names
)

pd.DataFrame({
    "iteration": np.arange(1, len(mlp.loss_curve_) + 1),
    "loss": mlp.loss_curve_
}).to_csv(OUTPUT_DIR / "lab1_loss_curve.csv", index=False)

ann_metrics_df.to_csv(OUTPUT_DIR / "lab1_split_metrics.csv", index=False)

print("Resultados guardados en:", OUTPUT_DIR)

## 11. Comparación con Lab 2 Task 1 y Task 2

Introduce las accuracies de test que aparecen al final de los notebooks de Lab 2. No se leen automáticamente porque las versiones actuales de esos notebooks no exportan una métrica final común a un archivo compartido. Se dejan como `None` para evitar utilizar accidentalmente resultados antiguos o inventados.

In [ ]:
# Sustituye los valores None por la accuracy de test de las ejecuciones
# de Lab 2 que quieras comparar. Ejemplo: 0.745
TASK1_TEST_ACCURACY = None  # CNN desde cero
TASK2_TEST_ACCURACY = None  # VGG16 transfer learning

comparison = pd.DataFrame([
    {"Modelo": "Lab 1 — ANN", "Test accuracy": ann_test_accuracy},
    {"Modelo": "Lab 2 Task 1 — CNN", "Test accuracy": TASK1_TEST_ACCURACY},
    {"Modelo": "Lab 2 Task 2 — VGG16", "Test accuracy": TASK2_TEST_ACCURACY},
])

display(comparison)

available = comparison.dropna(subset=["Test accuracy"])
if len(available) > 0:
    plt.figure(figsize=(9, 5))
    bars = plt.bar(available["Modelo"], available["Test accuracy"])
    plt.ylim(0, 1)
    plt.ylabel("Accuracy")
    plt.title("Comparación de accuracy en test")
    plt.xticks(rotation=15, ha="right")
    for bar, value in zip(bars, available["Test accuracy"]):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.015,
            f"{value:.3f}",
            ha="center"
        )
    plt.tight_layout()
    plt.show()
else:
    print("Introduce al menos una accuracy de Lab 2 para generar la gráfica.")

## 12. Interpretación y límites

- La accuracy de test estima el rendimiento sobre datos no utilizados para ajustar los pesos.
- La precision, recall y F1 macro dan el mismo peso a cada género.
- Una diferencia grande entre train y test puede indicar sobreajuste.
- La MLP recibe características tabulares; la CNN y VGG16 reciben imágenes. No tienen arquitecturas ni parámetros de optimización equivalentes.
- El esquema 64/16/20 y la semilla coinciden, pero no garantizan las mismas canciones exactas en cada split. Para una comparación emparejada habría que crear una partición por identificador de canción compartido y reutilizarla en los tres pipelines.

**Ruta de salida:** `MyDrive/MSLS/Results/lab1vslab2/`.